# Clasificación de estadios del sueño — Modelo CNN (Sleep-EDF Telemetry)

Este cuaderno implementa una Red Neuronal Convolucional (CNN) procesando directamente las series de tiempo crudas del EEG.

**Mejoras Críticas Implementadas:**
1. **Prevención de Data Leakage:** Uso de `GroupKFold` agrupando por sujeto (`S01`, `S02`...) para asegurar un esquema estricto Leave-One-Subject-Out (LOSO).
2. **Escalado Z-Score Intra-registro:** El EEG se normaliza independientemente por cada registro antes de entrar a la red, previniendo la explosión de gradientes.
3. **Gestión de Memoria (RAM):** Cast de los tensores gigantescos a `float32` para evitar crasheos en Colab (`OOM`).
4. **Prevención de Overfitting:** Incorporación de un *split* de validación interno y el callback `EarlyStopping`.
5. **Human-in-the-Loop:** Función extra para calcular la confianza de la predicción y enviar alertas de colores al Dashboard clínico.

In [1]:
# =========================================================
# INSTALACIÓN E IMPORTACIÓN DE DEPENDENCIAS
# =========================================================
!pip install --upgrade mne mlflow

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal
import mne
import mlflow

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import GroupKFold
from sklearn.metrics import cohen_kappa_score, f1_score, classification_report, confusion_matrix

mne.set_log_level("ERROR")
tf.get_logger().setLevel('ERROR')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuración de Hiperparámetros
El parámetro `N_SPLITS = 22` habilita automáticamente la evaluación LOSO (Leave-One-Subject-Out) completa, ya que la cohorte tiene exactamente 22 pacientes únicos.

In [29]:
BASE = Path("/content/drive/MyDrive/Bimestre 7/Proyectos/sleep-telemetry")
CANALES = ["EEG Fpz-Cz", "EEG Pz-Oz"]
EPOCA_SEG = 30.0
L_FREQ, H_FREQ = 0.3, 35.0
FUNDIR_N3_N4 = True

RECORTAR_W_MIN = 30
USAR_SUAVIZADO_HMM = True

# Configuración CNN
N_SPLITS = 22                # 22 para LOSO estricto
EPOCHS_MAX = 50
BATCH_SIZE = 64              # Ajustable dependiendo de la VRAM
LEARNING_RATE = 0.001

MAPA_ETAPAS = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N4",
    "Sleep stage R": "REM",
}
ETAPAS_VALIDAS = ["W", "N1", "N2", "N3", "REM"] if FUNDIR_N3_N4 \
    else ["W", "N1", "N2", "N3", "N4", "REM"]

## 2. Descubrimiento y Carga de Archivos
Se asegura la agrupación de la Noche 1 y Noche 2 del mismo paciente bajo el mismo `sujeto_id` para garantizar la validez clínica del experimento.

In [30]:
RE_ST = re.compile(r"^ST7(\d{2})(\d)J0-PSG\.edf$", re.IGNORECASE)

def descubrir_registros(base=BASE):
    base = Path(base)
    psgs = sorted(base.glob("*-PSG.edf"))
    registros = []

    for psg in psgs:
        m = RE_ST.match(psg.name)
        if m is None: continue

        ss, noche = m.group(1), m.group(2)
        candidatos = sorted(base.glob(f"ST7{ss}{noche}J?-Hypnogram.edf"))
        if not candidatos: continue

        registros.append({
            "psg": psg,
            "hyp": candidatos[0],
            "registro_id": f"ST7{ss}{noche}",
            "sujeto_id": f"S{ss}",
            "noche": int(noche),
        })
    return registros

def cargar_registro(psg_file, hyp_file, canales=CANALES):
    raw = mne.io.read_raw_edf(psg_file, preload=True, verbose="ERROR")
    disponibles = [c for c in canales if c in raw.ch_names]
    raw.pick(disponibles)
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
    ann = mne.read_annotations(hyp_file)
    return raw, ann, disponibles

## 3. Preprocesamiento Espacial y Escalado Crítico (Z-Score)
Es indispensable escalar los datos por registro antes de inyectarlos a la red neuronal. De lo contrario, los gradientes explotarán.

In [31]:
def etiquetas_por_epoca(ann, n_epocas, epoca_seg=EPOCA_SEG):
    etiquetas = np.full(n_epocas, None, dtype=object)
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        etapa = MAPA_ETAPAS.get(str(desc).strip())
        if not etapa: continue
        if FUNDIR_N3_N4 and etapa == "N4": etapa = "N3"
        i_ini = max(0, int(np.floor(onset / epoca_seg)))
        i_fin = min(n_epocas, int(np.ceil((onset + dur) / epoca_seg)))
        etiquetas[i_ini:i_fin] = etapa
    return etiquetas

def segmentar(datos, fs, epoca_seg=EPOCA_SEG):
    n_canales, n_total = datos.shape
    muestras_epoca = int(round(epoca_seg * fs))
    n_epocas = n_total // muestras_epoca
    datos = datos[:, :n_epocas * muestras_epoca]
    epocas = datos.reshape(n_canales, n_epocas, muestras_epoca)
    return np.transpose(epocas, (1, 0, 2))

def indices_recorte(y, minutos=RECORTAR_W_MIN, epoca_seg=EPOCA_SEG):
    if minutos is None: return 0, len(y)
    n_margen = int(minutos * 60 / epoca_seg)
    es_sueno = np.array([e in ("N1", "N2", "N3", "REM") for e in y])
    if not es_sueno.any(): return 0, len(y)
    primero = int(np.argmax(es_sueno))
    ultimo = len(y) - 1 - int(np.argmax(es_sueno[::-1]))
    return max(0, primero - n_margen), min(len(y), ultimo + 1 + n_margen)

def procesar_registro_cnn(reg):
    raw, ann, canales = cargar_registro(reg["psg"], reg["hyp"])
    fs = raw.info["sfreq"]

    epocas_data = segmentar(raw.get_data() * 1e6, fs)
    y = etiquetas_por_epoca(ann, n_epocas=epocas_data.shape[0])

    i0, i1 = indices_recorte(y)
    epocas_data = epocas_data[i0:i1]
    y = y[i0:i1]

    valido = np.array([e in ETAPAS_VALIDAS for e in y])
    epocas_data = epocas_data[valido]
    y = y[valido]

    # CORRECCIÓN CRÍTICA: Escalado Z-Score por registro (evita explotar la red)
    media = np.mean(epocas_data, axis=2, keepdims=True)
    desviacion = np.std(epocas_data, axis=2, keepdims=True) + 1e-8
    epocas_data = (epocas_data - media) / desviacion

    # Reorganizar a (n_epocas, n_muestras, n_canales)
    X_cnn = np.transpose(epocas_data, (0, 2, 1))

    label_map = {stage: i for i, stage in enumerate(ETAPAS_VALIDAS)}
    y_cnn = to_categorical([label_map[stage] for stage in y], num_classes=len(ETAPAS_VALIDAS))

    meta = pd.DataFrame({
        "sujeto_id": reg["sujeto_id"],
        "registro_id": reg["registro_id"],
        "noche": reg["noche"],
    }, index=range(len(y)))

    return X_cnn, y_cnn, meta

def construir_dataset_cnn(registros):
    Xs, ys, ms = [], [], []
    for reg in registros:
        X, y, m = procesar_registro_cnn(reg)
        Xs.append(X)
        ys.append(y)
        ms.append(m)

    # CORRECCIÓN DE RAM: Casteo a float32 para evitar OOM (Out of Memory)
    X_final = np.concatenate(Xs, axis=0).astype(np.float32)
    y_final = np.concatenate(ys, axis=0).astype(np.float32)
    meta_final = pd.concat(ms, ignore_index=True)
    return X_final, y_final, meta_final

## 4. Arquitectura CNN
Diseño de red base con capas convolucionales y de MaxPooling intercaladas con regularización por Dropout para prevenir el sobreajuste.

In [40]:
def create_cnn_model(input_shape, num_classes, filtros=[64, 128], dropout_rate=0.5, learning_rate=0.001):
    input_layer = Input(shape=input_shape)

    x = Conv1D(filters=filtros[0], kernel_size=10, activation='relu', padding='same')(input_layer)
    x = MaxPooling1D(pool_size=8)(x)
    x = Dropout(dropout_rate)(x)

    x = Conv1D(filters=filtros[1], kernel_size=8, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=4)(x)
    x = Dropout(dropout_rate)(x)

    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(dropout_rate)(x)

    output_layer = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output_layer)
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

## 5. Módulos de Evaluación, HMM (Viterbi) y Confianza Clínica
Se incorpora la función de cálculo de nivel de confianza. Esto habilitará al Dashboard a semaforizar las alertas de la interfaz gráfica

In [41]:
def matriz_transicion(secuencias, clases, suavizado=1.0):
    idx = {c: i for i, c in enumerate(clases)}
    K = len(clases)
    A, pi = np.full((K, K), suavizado), np.full(K, suavizado)

    for seq in secuencias:
        if not seq: continue
        pi[idx[seq[0]]] += 1
        for a, b in zip(seq[:-1], seq[1:]):
            A[idx[a], idx[b]] += 1
    return A / A.sum(axis=1, keepdims=True), pi / pi.sum()

def viterbi(log_emision, A, pi):
    n, K = log_emision.shape
    logA, logpi = np.log(A + 1e-12), np.log(pi + 1e-12)
    delta, psi = np.zeros((n, K)), np.zeros((n, K), dtype=int)
    delta[0] = logpi + log_emision[0]

    for t in range(1, n):
        m = delta[t - 1][:, None] + logA
        psi[t] = np.argmax(m, axis=0)
        delta[t] = m[psi[t], np.arange(K)] + log_emision[t]

    camino = np.zeros(n, dtype=int)
    camino[-1] = int(np.argmax(delta[-1]))
    for t in range(n - 2, -1, -1):
        camino[t] = psi[t + 1, camino[t + 1]]
    return camino

def suavizar_hmm(proba, clases, A, pi, prior, registros_epoca):
    pred = np.empty(len(proba), dtype=object)
    log_emision = np.log(proba + 1e-12) - np.log(prior + 1e-12)

    for rid in pd.unique(registros_epoca):
        m = (registros_epoca == rid).to_numpy()
        camino = viterbi(log_emision[m], A, pi)
        pred[m] = [clases[i] for i in camino]
    return pred

def generar_predicciones_con_confianza(proba, clases, umbral_duda=0.60):
    """Genera la semaforización para la UI basada en el margen softmax"""
    pred_idx = np.argmax(proba, axis=1)
    confianzas = np.max(proba, axis=1)

    resultados = []
    for i, idx in enumerate(pred_idx):
        conf = confianzas[i]
        alerta = conf < umbral_duda

        # Semaforización para tablero Human-in-the-Loop
        if conf >= 0.75: color = "verde"
        elif conf >= umbral_duda: color = "amarillo"
        else: color = "rojo"

        resultados.append({
            "epoca": i,
            "estadio": clases[idx],
            "confianza": round(float(conf), 3),
            "alerta_revision": alerta,
            "color_ui": color
        })

    return pd.DataFrame(resultados)

## 6. Bucle de Entrenamiento (GroupKFold + Early Stopping)
Este pipeline preserva la integridad inter-sujeto mediante `GroupKFold` y detiene el entrenamiento tempranamente gracias a `EarlyStopping`, evaluando constantemente el `val_loss`.

In [44]:
def entrenar_cv_cnn(X, y, meta, filtros=[64,128], dropout=0.5, lr=0.001, batch=64):
    grupos = meta["sujeto_id"].to_numpy()
    n_sujetos = len(np.unique(grupos))
    n_splits_reales = min(N_SPLITS, n_sujetos)

    print(f"\n[Validación] GroupKFold: {n_splits_reales} folds sobre {n_sujetos} sujetos (LOSO)\n")
    gkf = GroupKFold(n_splits=n_splits_reales)
    clases_str = np.array(ETAPAS_VALIDAS)

    y_true_all_numeric, y_pred_all_numeric, y_pred_hmm_all_numeric = [], [], []
    kappas_crudo, kappas_hmm = [], []

    # ---> LÍNEA RESTAURADA: Extraer dinámicamente las dimensiones (3000, 2)
    input_shape = X.shape[1:]

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    for fold, (i_tr, i_te) in enumerate(gkf.split(X, y, groups=grupos), 1):
        print(f"  --> Entrenando Fold {fold}/{n_splits_reales}...")

        # La función ahora recibe input_shape correctamente
        cnn_model = create_cnn_model(input_shape, len(clases_str), filtros=filtros, dropout_rate=dropout, learning_rate=lr)

        historial = cnn_model.fit(
            X[i_tr], y[i_tr],
            epochs=EPOCHS_MAX,
            batch_size=batch,
            validation_split=0.15,
            callbacks=[early_stopping],
            verbose=1
        )

        proba = cnn_model.predict(X[i_te], verbose=0)
        pred_numeric = np.argmax(proba, axis=1)
        y_true_numeric = np.argmax(y[i_te], axis=1)

        meta_tr = meta.iloc[i_tr]
        y_train_text = np.array([clases_str[np.argmax(label_onehot)] for label_onehot in y[i_tr]])
        secuencias = [y_train_text[meta_tr["registro_id"] == r].tolist() for r in pd.unique(meta_tr["registro_id"])]

        A, pi = matriz_transicion(secuencias, list(clases_str))
        prior_dict = pd.Series(y_train_text).value_counts(normalize=True).reindex(clases_str, fill_value=1e-12).to_dict()
        prior = np.array([prior_dict[c] for c in clases_str])

        pred_hmm_text = suavizar_hmm(
            proba, list(clases_str), A, pi, prior,
            meta.iloc[i_te]["registro_id"].reset_index(drop=True),
        ) if USAR_SUAVIZADO_HMM else clases_str[pred_numeric]

        label_map_inv = {stage: i for i, stage in enumerate(clases_str)}
        pred_hmm_numeric = np.array([label_map_inv[stage] for stage in pred_hmm_text])

        k_c = cohen_kappa_score(y_true_numeric, pred_numeric)
        k_h = cohen_kappa_score(y_true_numeric, pred_hmm_numeric)
        kappas_crudo.append(k_c)
        kappas_hmm.append(k_h)
        print(f"      Resultado Fold {fold}: Kappa crudo = {k_c:.4f} | con HMM = {k_h:.4f}")

        y_true_all_numeric.extend(y_true_numeric)
        y_pred_all_numeric.extend(pred_numeric)
        y_pred_hmm_all_numeric.extend(pred_hmm_numeric)

        if fold == n_splits_reales:
            df_confianza = generar_predicciones_con_confianza(proba, clases_str)

    print(f"\nKappa Global Crudo: {np.mean(kappas_crudo):.4f} +/- {np.std(kappas_crudo):.4f}")
    print(f"Kappa Global HMM  : {np.mean(kappas_hmm):.4f} +/- {np.std(kappas_hmm):.4f}")

    return y_true_all_numeric, y_pred_hmm_all_numeric, df_confianza, historial

## 7. Ejecución Principal
Este bloque ensambla todo el entorno. Si tienes AWS EC2 levantado, puedes configurar la URI de MLflow aquí.

In [ ]:
import os
import mlflow
import tensorflow as tf

# --- PANEL DE CONTROL MLOPS (Corrida: Filtros Profundos) ---
VAR_RUN_NAME = "cnn_lr_lento"
VAR_FILTROS  = [64, 128]  # Profundidad duplicada para capturar husos del sueño
VAR_DROPOUT  = 0.5
VAR_LR       = 0.0001
VAR_BATCH    = 64

# 1. Limpiar memoria de la GPU
tf.keras.backend.clear_session()

# 2. Configurar la ruta de guardado ANTES de iniciar el registro
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
# NOTA: Usamos la ruta pura sin 'file://' para evitar el error INTERNAL_ERROR por el espacio
mlflow.set_tracking_uri("/content/drive/MyDrive/Bimestre 7/Proyectos/mlruns")
experiment = mlflow.set_experiment("/modelo_cnn_colab")

print("Descubriendo registros EDFx...")
registros = descubrir_registros(BASE)

print("\nConstruyendo dataset masivo (segmentación y escalado Z-score)...")
X_cnn, y_cnn, meta_cnn = construir_dataset_cnn(registros)
print(f"Dataset Formateado CNN: {X_cnn.shape[0]} épocas x {X_cnn.shape[1]} muestras x {X_cnn.shape[2]} canales")

# 3. Iniciar UN SOLO registro de MLflow con el nuevo nombre
with mlflow.start_run(run_name=VAR_RUN_NAME):

    # Consolidamos todos los parámetros en un solo envío
    mlflow.log_params({
        "modelo": "CNN_TinySleepNet_Profunda",
        "filtros": str(VAR_FILTROS),
        "learning_rate": VAR_LR,
        "batch_size": VAR_BATCH,
        "dropout": VAR_DROPOUT,
        "n_splits_loso": N_SPLITS,
        "n_epocas_total": X_cnn.shape[0]
    })

    # 4. Ejecución del entrenamiento
    y_true_global, y_pred_global, df_alertas_dashboard, historial_entrenamiento = entrenar_cv_cnn(
        X_cnn, y_cnn, meta_cnn,
        filtros=VAR_FILTROS,
        dropout=VAR_DROPOUT,
        lr=VAR_LR,
        batch=VAR_BATCH
    )

    print("\nVisualización de Confianza Clínica (Ejemplo último Fold para Interfaz Visual):")
    display(df_alertas_dashboard.head(15))

Descubriendo registros EDFx...

Construyendo dataset masivo (segmentación y escalado Z-score)...
Dataset Formateado CNN: 42467 épocas x 3000 muestras x 2 canales

[Validación] GroupKFold: 22 folds sobre 22 sujetos (LOSO)

  --> Entrenando Fold 1/22...
Epoch 1/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.5249 - loss: 1.1824 - val_accuracy: 0.6296 - val_loss: 1.0206
Epoch 2/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.6850 - loss: 0.8141 - val_accuracy: 0.7042 - val_loss: 0.8450
Epoch 3/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7336 - loss: 0.7151 - val_accuracy: 0.5804 - val_loss: 0.9038
Epoch 4/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7590 - loss: 0.6578 - val_accuracy: 0.6701 - val_loss: 0.7989
Epoch 5/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7733 - loss: 0.6197 - val_accuracy: 0.7032 - val_loss: 0.7563
Epoch 6/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7842 - loss: 0.5919 - val_accuracy: 0.71

## 8. Evaluación y Visualización de Resultados

Análisis global del rendimiento de la red neuronal convolucional (CNN) tras la validación cruzada LOSO. Se incluye la matriz de confusión normalizada (Recall) para evaluar el desempeño en la clase minoritaria (N1) y el reporte de clasificación estandarizado.

In [ ]:
# =========================================================
# 8. VISUALIZACIÓN DE RESULTADOS Y MATRIZ DE CONFUSIÓN
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 1. Recuperar los datos de la ejecución (asegúrate de desempaquetar las 3 variables en la Sección 7)
# y_true_global, y_pred_global, df_alertas_dashboard = entrenar_cv_cnn(...)

clases_str = ["W", "N1", "N2", "N3", "REM"]

# 2. Calcular la Matriz de Confusión
cm = confusion_matrix(y_true_global, y_pred_global)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # Normalizada por fila (Recall)

# 3. Graficar el Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=clases_str, yticklabels=clases_str,
            vmin=0, vmax=1)
plt.title("Matriz de Confusión Normalizada (CNN + HMM)")
plt.ylabel("Estadio Real (Anotación Experta)")
plt.xlabel("Estadio Predicho (IA)")
plt.show()

# 4. Reporte de Clasificación Textual
print("\nReporte de Clasificación Global:")
print(classification_report(y_true_global, y_pred_global, target_names=clases_str))

### 8.1 Comparativa de Arquitectura del Sueño (Hipnograma)

Visualización del efecto del modelo predictivo sobre la continuidad del sueño. Se compara el hipnograma real anotado por el técnico frente a la predicción final de la CNN suavizada con Cadenas de Markov (Viterbi HMM).

In [ ]:
# Graficamos una porción del registro (ej. 1000 épocas = aprox. 8 horas de sueño)
limite = min(1000, len(y_true_global))

# Mapeo visual de índices numéricos al estándar médico del eje Y
# Índices del modelo: 0=W, 1=N1, 2=N2, 3=N3, 4=REM
# Coordenadas Y médicas (arriba hacia abajo): W=4, REM=3, N1=2, N2=1, N3=0
mapa_visual = {0: 4, 1: 2, 2: 1, 3: 0, 4: 3}

y_real_num = [mapa_visual[e] for e in y_true_global[:limite]]
y_pred_num = [mapa_visual[e] for e in y_pred_global[:limite]]

# Convertir épocas de 30 segundos a escala de horas
tiempo = np.arange(len(y_real_num)) / 120

fig, ax = plt.subplots(figsize=(15, 4))

# Hipnograma Real (Gris) vs Predicho (Azul)
ax.step(tiempo, y_real_num, label="Real (Anotación Experta)", color="gray", linewidth=3, alpha=0.6)
ax.step(tiempo, y_pred_num, label="Predicción (CNN + HMM)", color="#1f77b4", linewidth=2)

ax.set_yticks([0, 1, 2, 3, 4])
ax.set_yticklabels(["N3", "N2", "N1", "REM", "W"])
ax.set_xlabel("Tiempo (Horas)")
ax.set_title("Comparativa de Hipnogramas: Diagnóstico Real vs. IA")
ax.legend(loc="upper right")
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 8.2 Curvas de Aprendizaje (Loss y Accuracy)

Monitoreo de la convergencia de la red en el último *fold* de validación cruzada. Evidencia gráfica de la intervención del *Early Stopping* para prevenir el sobreajuste al detener el entrenamiento cuando el `val_loss` deja de mejorar.

In [ ]:
# Asegúrate de desempacar la variable del historial en la Celda 16:
# y_true_global, y_pred_global, df_alertas, historial_entrenamiento = entrenar_cv_cnn(...)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Gráfica de Pérdida (Loss)
ax1.plot(historial_entrenamiento.history['loss'], label='Entrenamiento (Train Loss)', color='#1f77b4', linewidth=2)
ax1.plot(historial_entrenamiento.history['val_loss'], label='Validación (Val Loss)', color='#ff7f0e', linewidth=2, linestyle='--')
ax1.set_title('Convergencia del Modelo (Loss)')
ax1.set_xlabel('Épocas de Entrenamiento')
ax1.set_ylabel('Categorical Crossentropy')
ax1.legend()
ax1.grid(alpha=0.3)

# Gráfica de Exactitud (Accuracy)
ax2.plot(historial_entrenamiento.history['accuracy'], label='Entrenamiento (Train Acc)', color='#2ca02c', linewidth=2)
ax2.plot(historial_entrenamiento.history['val_accuracy'], label='Validación (Val Acc)', color='#d62728', linewidth=2, linestyle='--')
ax2.set_title('Rendimiento del Modelo (Accuracy)')
ax2.set_xlabel('Épocas de Entrenamiento')
ax2.set_ylabel('Exactitud')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 8.3 Análisis de Falsos Positivos en la Fase de Transición (N1)

Desglose de los errores de clasificación en el estadio N1 (clase minoritaria). Permite justificar clínicamente si las confusiones del modelo ocurren hacia estadios adyacentes lógicos (como vigilia W o sueño ligero N2).

In [ ]:
# Extraemos la fila correspondiente a N1 de la matriz de confusión (índice 1)
# Índices: 0=W, 1=N1, 2=N2, 3=N3, 4=REM
fila_n1 = cm[1, :]

# Excluimos los aciertos (N1 predicho como N1) para graficar únicamente los errores
errores_n1 = [fila_n1[0], fila_n1[2], fila_n1[3], fila_n1[4]]
etiquetas_errores = ["Falso W", "Falso N2", "Falso N3", "Falso REM"]
colores = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd"]

plt.figure(figsize=(8, 5))
bars = plt.bar(etiquetas_errores, errores_n1, color=colores, alpha=0.8)

# Agregar los valores exactos sobre cada barra
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max(errores_n1)*0.02),
             int(yval), ha='center', va='bottom', fontweight='bold')

plt.title("Desglose de Confusiones Clínicas para el Estadio N1")
plt.ylabel("Número de Épocas Mal Clasificadas")
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 9. Interfaz Gráfica de MLflow (Dashboard)

Despliegue del servidor web de MLflow directamente en el entorno de Google Colab. Esta interfaz permite monitorear el registro histórico, comparar visualmente los hiperparámetros de las 10 corridas planificadas y determinar qué configuración maximiza la precisión clínica del modelo.

In [26]:
# =========================================================
# 9. INTERFAZ GRÁFICA DE MLFLOW (CORRECCIÓN DE RUTA)
# =========================================================
import subprocess
import time
import re

# 1. Limpiar procesos
!pkill -f mlflow
!pkill -f cloudflared

# 2. RUTA CORREGIDA: Sin el prefijo "file://" para evitar errores con los espacios
tracking_uri = "/content/drive/MyDrive/Bimestre 7/Proyectos/mlruns"

# 3. Levantar MLflow
subprocess.Popen(["mlflow", "ui", "--backend-store-uri", tracking_uri, "--port", "5000", "--host", "0.0.0.0"])
time.sleep(3)

# 4. Crear túnel Cloudflare
print("Iniciando túnel seguro de Cloudflare...\n")
proceso_cf = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000', '--http-host-header', '127.0.0.1'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# 5. Extraer enlace
for line in proceso_cf.stdout:
    linea = line.decode('utf-8', errors='replace')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', linea)
    if match:
        print("\n" + "="*75)
        print("✅ CONEXIÓN ESTABLE. HAZ CLIC EN ESTE ENLACE:")
        print(match.group(0))
        print("="*75)
        break

Iniciando túnel seguro de Cloudflare...


✅ CONEXIÓN ESTABLE. HAZ CLIC EN ESTE ENLACE:
https://procedures-witnesses-ratios-evaluated.trycloudflare.com
